In [2]:
import numpy as np
import setup as stp
import Maxwell2D as Max
import mesh_reader as msh
import matplotlib.pyplot as plt

N = 1

# 1. Insira os dados coletados das suas 4 malhas
pontos_borda = np.array([11, 21, 41, 81])

erros = []
for n in range(len(pontos_borda)):
    VX, VY, EToV, BCTags = msh.MeshReader2D(f'malhas/h{n}.msh')

    malha = stp.StartUp2D(N,EToV,VX,VY)

    area_total = np.sum(malha.J)/(2*malha.N)
    print(f"Área total do domínio Numérico: {area_total:.10f}")
    print((2*np.pi)**2)

    Ez = np.zeros((malha.Np,malha.K),dtype=np.float64)
    Ez = np.sin(malha.x)*np.sin(malha.y)
    Hx = np.zeros((malha.Np,malha.K), dtype=np.float64)
    Hy = np.zeros((malha.Np,malha.K),dtype= np.float64)

    #FinalTime = 3 * np.pi * np.sqrt(2)
    FinalTime = 5

    Hx, Hy, Ez, pp, t, erro = Max.Maxwell2D(Hx,Hy,Ez,FinalTime,malha)

    Ez_analitico = np.sin(malha.x)*np.sin(malha.y)*np.cos(np.sqrt(2)*t[-1])

    M = np.linalg.inv(malha.V @ malha.V.T) 

    erro_nodal = Ez - Ez_analitico

    integral_erro_sq = np.sum(malha.J * (erro_nodal * (M @ erro_nodal)))
    En = np.sqrt(integral_erro_sq)

    integral_analitico_sq = np.sum(malha.J * (Ez_analitico * (M @ Ez_analitico)))
    wt = np.sqrt(integral_analitico_sq)

    erro_L2_real = En / wt

    erros.append(erro_L2_real)

Malha carregada com sucesso!
Número de vértices: 121
Número de Elementos (K): 200
Área total do domínio Numérico: 29.6088132033
39.47841760435743
Tempo atual: 7.0833e-02 / 5.0000e+00
Tempo atual: 1.4167e-01 / 5.0000e+00
Tempo atual: 2.1250e-01 / 5.0000e+00
Tempo atual: 2.8333e-01 / 5.0000e+00
Tempo atual: 3.5417e-01 / 5.0000e+00
Tempo atual: 4.2500e-01 / 5.0000e+00
Tempo atual: 4.9583e-01 / 5.0000e+00
Tempo atual: 5.6667e-01 / 5.0000e+00
Tempo atual: 6.3750e-01 / 5.0000e+00
Tempo atual: 7.0833e-01 / 5.0000e+00
Tempo atual: 7.7917e-01 / 5.0000e+00
Tempo atual: 8.5000e-01 / 5.0000e+00
Tempo atual: 9.2083e-01 / 5.0000e+00
Tempo atual: 9.9167e-01 / 5.0000e+00
Tempo atual: 1.0625e+00 / 5.0000e+00
Tempo atual: 1.1333e+00 / 5.0000e+00
Tempo atual: 1.2042e+00 / 5.0000e+00
Tempo atual: 1.2750e+00 / 5.0000e+00
Tempo atual: 1.3458e+00 / 5.0000e+00
Tempo atual: 1.4167e+00 / 5.0000e+00
Tempo atual: 1.4875e+00 / 5.0000e+00
Tempo atual: 1.5583e+00 / 5.0000e+00
Tempo atual: 1.6292e+00 / 5.0000e+00
Tem

In [ ]:
# Cálculo do tamanho h real baseado no tamanho da cavidade
L_total = 2 * np.pi
h_valores = L_total / (pontos_borda - 1)

# SUBSTITUA AQUI pelos erros reais que o seu código cuspir rodando com N=1!
# Exemplo fictício de queda em ordem h^2:
erros = np.array(erros) 

# ------------------------------------------------------------------
# 2. Cálculo Matemático da Ordem (k)
# ------------------------------------------------------------------
print("Ordem de Convergência Espacial (k):")
for i in range(len(h_valores) - 1):
    h_antigo = h_valores[i]
    h_novo = h_valores[i+1]
    
    E_antigo = erros[i]
    E_novo = erros[i+1]
    
    # A fórmula matemática para a inclinação no gráfico Log-Log
    ordem_k = (np.log(E_novo) - np.log(E_antigo)) / (np.log(h_novo) - np.log(h_antigo))
    
    print(f"De {pontos_borda[i]:2d} para {pontos_borda[i+1]:2d} pontos -> Ordem = {ordem_k:.2f}")

# ------------------------------------------------------------------
# 3. Gráfico Log-Log (Estilo Livro-Texto)
# ------------------------------------------------------------------
# Ativa o renderizador do LaTeX para ficar bonito
plt.rcParams['text.usetex'] = True

plt.figure(figsize=(8, 6))

# Usa loglog para os dois eixos!
plt.loglog(h_valores, erros, marker='o', linestyle='-', linewidth=2, 
           markersize=8, color='royalblue', label=r'Erro Numérico ($N=1$)')

# Adiciona uma "Reta Guia" teórica para a banca ver que a inclinação bate
# Multiplicamos por uma constante apenas para transladar a reta e deixá-la visível ao lado
constante_guia = erros[0] / (h_valores[0]**2)
plt.loglog(h_valores, constante_guia * (h_valores**2), 
           linestyle='--', color='gray', label=r'Referência Teórica $\mathcal{O}(h^2)$')

plt.xlabel(r"Tamanho característico do elemento $h$ (Escala Log)", fontsize=12)
plt.ylabel(r"Norma $||E_z - E_{z,h}||_{L^2}$ (Escala Log)", fontsize=12)
plt.title(r"\textbf{Convergência Espacial ($h$-refinement)}", fontsize=14)

plt.grid(True, which="both", ls="--", alpha=0.3)
plt.legend(fontsize=12)
plt.show()

Ordem de Convergência Espacial (k):
De 11 para 21 pontos -> Ordem = 2.28
De 21 para 41 pontos -> Ordem = 2.10
De 41 para 81 pontos -> Ordem = 2.02


RuntimeError: Failed to process string with tex because latex could not be found

<Figure size 800x600 with 1 Axes>